<a href="https://colab.research.google.com/github/Karen-Kwatia/lab-4-llm-decision-support/blob/main/Lab4_llm_decision_support.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:

# API-key setup — DO NOT hard-code your key in this cell.

import os
# --- Google Colab (Secrets panel) ---
# TODO: set API_KEY using ONE of the methods above.
from google.colab import userdata
API_KEY = userdata.get("GROQ_API_KEY")


# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "llama-3.3-70b-versatile"

print("Client ready.")

Client ready.


SECTION 1-TALKING TO AN LLM PROGRAMMATICALLY


PART 1.1 -YOUR FIRST API CALL

In [4]:
# TODO: Write a helper function you will reuse for the WHOLE lab:


def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    return response.choices[0].message.content

# TODO: Call it once with a simple question and print the answer.
expected_answer=ask_llm("What is the name of the president of Ashesi University?")
print(expected_answer)

# TODO: Print response.usage as well — how many tokens did your call consume?
response_usage=  client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is the name of the president of Ashesi University?"}],
)
print(response_usage)


The founder and president of Ashesi University is Patrick Awuah.
ChatCompletion(id='chatcmpl-b69bd663-dcce-4f12-8b99-4cf528f419e3', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The founder and president of Ashesi University is Patrick Awuah.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1786471377, model='llama-3.3-70b-versatile', object='chat.completion', moderation=None, service_tier='on_demand', system_fingerprint='fp_dae98b5ecb', usage=CompletionUsage(completion_tokens=15, prompt_tokens=47, total_tokens=62, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.094000813, prompt_time=0.01207939, completion_time=0.082330743, total_time=0.094410133), usage_breakdown=None, x_groq={'id': 'req_01kzrzvtfaewgrpjr8zpkhab2e', 'seed': 1148512040})


STUDENT REASONING
1. The system gives instructions to AI model before conversations start. It sets boundaries on what the model can do and how its personality should look like.
The user is the human operator that is going to use the model,the user submits questions and tasks to the model and expects results at the end.

2.A token is an atomic unit of the question presented to the AI model.API providers bill per token rather than per requests because of different request lengths and the computations it has to compute. For example, building by requests would mean the question;"Who is the president of Ashesi University?" and the question " Using Ashesi University as a reference point, write a 5000 research paper on the university, the president, staff members, community and impact. Include informations about the year of foundation, its mission and values, alumni and current students, age range of students, careers of alumni, founding partners and scholarships available" would have the same price. But this is not fair, the second question is more longer and has more tokens in its request than the first question and would use more computations compared to the first one. API providers bill per tokens rather than requests to differentiate between the amount of work done. Longer requests with more tokens have higher computational costs than shorter requests with small number of tokens.

PART 1.2 -TEMPERATURE :THE RANDOMNESS DIAL


In [5]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."
#Asking the same question when temperature is 0.0
print("TEMPERATURE IS 0.0")

for i in range(5):
    answer_to_question=ask_llm("Suggest a name for a savings product for market traders in Accra.",temperature=0.0)
    print(answer_to_question)
print()

print("TEMPERATURE IS 1.2")
for i in range(5):
    answer_to_question=ask_llm("Suggest a name for a savings product for market traders in Accra.",temperature=1.2)
    print(answer_to_question)
print()
# TODO: Print all 10 answers, grouped by temperature.


TEMPERATURE IS 0.0
Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Accra Amanfu**: "Amanfu" is a Ghanaian word for "savings" or "treasury", so this name incorporates local language and culture.
4. **Market Mobi**: This name is short and catchy, and "Mobi" implies mobility and flexibility, which could appeal to market traders who need to manage their finances on-the-go.
5. **Sika Su**: "Sika" is the Ghanaian word for "money", and "Su" means "grow" or "increase", so this name suggests a savings product that helps traders grow their wealth.
6. **Kokroko Savings**: "Kokroko" is a Ghanaian word for "honest" or "trustworthy", which could convey a sense of reliability and security for market traders.
7. **Traders' Trust**: This name emphasizes the idea of tru

When the temperature was 0.0 the model was predictable and repetitive. Most of the names like "Makola Save","Sika Saver", appeared across most of the answers and when the temperature was 1.2, the responses were varied and different names were generated. The difference between the temperatures is that when the temperature is high, the model gives varying answers and is unpredictable comapre to when it is low.

I think the temperature appropriate for the loan decision is the support system is 0.0 or a temperature close to 0.0. This is because loan decisions should be predictable and consistent. Increasing the temperature introduces unpredicatbility and randomness which is not fair. An approval of a loan decision should be consistent and not random.

SECTION 2-The Dataset: Loan Application Letters


In [6]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


SECTION 3 -Prompt Engineering for the Decision Support System

Part 3.1 — Component 1: Summarization

In [7]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.
print("NAIVE ATTEMPT")
SUMMARY_PROMPT_V1="Summarize this"
print(ask_llm(f"{SUMMARY_PROMPT_V1} {LETTERS['L002']}"))
print(ask_llm(f"{SUMMARY_PROMPT_V1} {LETTERS['L006']}"))
print()

# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
print("PROPER TEMPLATE ATTEMPT")
SUMMARY_PROMPT_V2="You are an assistant to a microfinance loan officer and you have received all these applications for loans.You are to review these applications. In your review, be factual, neutral and do not add any invented details and summarize these applications in 3-4 sentences "
print(ask_llm(f"{SUMMARY_PROMPT_V2} {LETTERS['L002']}"))

print(ask_llm(f"{SUMMARY_PROMPT_V2} {LETTERS['L006']}"))
print()




#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
user_prompt_template ="Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.
print(ask_llm(user_prompt_template.format(letter_text=LETTERS['L002']),system_prompt=SUMMARY_PROMPT_V2,temperature=0))
print()
print(ask_llm(user_prompt_template.format(letter_text=LETTERS['L006']),system_prompt=SUMMARY_PROMPT_V2,temperature=0))
print()

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.

NAIVE ATTEMPT
Kwame Boateng, a commercial driver in Kumasi, is seeking a loan of GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but expects it to improve after the festive season. He doesn't have collateral, but is asking for help and promising to repay the loan when he can.
Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. Although he has no experience and no collateral, he claims to be "business-minded" based on his friends' opinions and promises to repay the loan within a year when his businesses are supposedly thriving.

PROPER TEMPLATE ATTEMPT
Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000 to repair his trotro engine and settle personal debts. He mentions that business has been slow, but expects it to improve after the festive season. Kwame does not have collateral to offer

VI's output was vague and loses some details. For example in L001,Kwame Boaateng is seeking the loan to pay for his trotro engine. V1 does not state this but just said Kwame is seeking the loan to pay for his vehicle,without stating the specific type of vehicle that Kwame is going to fix the engine of. V2 corrects this by specifying the type of vehicle engine, Kwame wants to fix.
V1's output has no restriction on the length and therefore runs quite loosely but V2's output has a constraint and hence the effect of that length constraint is seen and used.
Also, in V1 the language is interpretive whiles V2 remains neutral.For example in V1(L006) it says "he claims to be business minded" which is drawing meaning from the application letter but V2(L006) remain neutral by not drawing any meaning from the application letter.

QUESTION 2
No invented details is essential in this application because the loan is given based on merits and the application letter of each applicant contains the merit that the applicant has.If the model adds details that do not exist, it affects the decisions of the loan officer and the loan might be given to someone who does not have enough resources to pay.The failure mode of this is called hallucination. Hallucinations occur when the AI model generates information that is false, misleading or fabricated and presents it as though it is correct. When this occurs, the loan office cannot detect that this is false information and the purpose of the summary fails and fairness and accuracy is not achieved.

PART 3.2-Component 2: Structured extraction (JSON)


In [8]:
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON

#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0
import json
import pandas as pd

extract_prompt = """You are an experienced data extraction agent for a microfinance loan agency. You are to
read the letter below and extract the following details from the letter:
1. applicant_name (string) - the name of the applicant
2. amount_ghs (number) - the amount the applicant is requesting for
3. purpose (string) - what the applicant is going to use the money for
4. monthly_profit_ghs (number or null) - the applicant's monthly profit, if not stated use null
5. has_collateral_or_guarantor (boolean) - whether the applicant has a collateral or a guarantor (true if there is a collateral/guarantor and false otherwise)
6. repayment_months (number or null) - how many months it will take to repay, if the applicant stated it

RULES
1. Do not mention any field that is not stated in the letter
2. Do not make any inference or guesses
3. If a field is not stated in the letter, use null
4. Return ONLY a JSON object with EXACTLY these keys:
applicant_name (string), amount_ghs (number), purpose (string),
monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
repayment_months (number or null)

EXAMPLE:
Hello, my name is Dentaa. I am a twenty-two year old sales assistant at the Accra Mall. Currently, I am earning 1200 cedis every month.
I am applying for a 5000 cedis loan to help me get a proper phone and content creation equipment to help me start my content creation.
I am expecting to earn approximately 1000 cedis from content creation at the end of every month and I expect to pay within 12 months.
I have spoken to my aunt who has agreed to be my guarantor to help me get this loan. Thank you for your time.

EXPECTED OUTPUT:
{{
  "applicant_name": "Dentaa",
  "amount_ghs": 5000,
  "purpose": "To start content creation",
  "monthly_profit_ghs": 1200,
  "has_collateral_or_guarantor": true,
  "repayment_months": 12
}}

Now extract the fields from this letter:

{letter_text}
"""



# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).

def extract_fields(letter_text):
  extracted_output=ask_llm(extract_prompt.format(letter_text=letter_text),temperature=0)

  clean_output = extracted_output.strip().strip("`").replace("json","",1).strip()

  try:
    return json.loads(clean_output)
  except:
    print("WARNING-Failure occured")
    return None

# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.
collect_results=[]
for letter in LETTERS:
  collect_results.append(extract_fields(LETTERS[letter]))
df =pd.DataFrame(collect_results)
df




,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,Akosua Mensah,8000,to buy a deep freezer and expand into frozen f...,900.0,True,20.0
1,Kwame Boateng,25000,to repair my trotro engine and settle some per...,NaN,False,NaN
2,Efua Darko,15000,to purchase two industrial sewing machines and...,2800.0,True,15.0
3,Yaw Owusu,12000,for feed and 500 new layers for my poultry farm,1500.0,True,18.0
4,Adenta Women's Weaving Cooperative,30000,to buy a bulk order of yarn directly from the ...,NaN,True,16.0
5,Kofi,50000,"to start a car washing business, a provision s...",NaN,False,12.0
